# Exploring the EnergyGuard ontology with Python

No Cypher needed. The ontology is a **catalogue**: it says which datasets and variables exist for each pilot and where
their numbers live in the data lake. It does not contain the measurements themselves, and **this notebook never connects
to the data lake**; it only reads the ontology (step 8 prints the SQL you would run there).

Everything hangs off a tree: **pilot → dataset → variable**, and each variable has a measured **measurement type** (Temperature, Pressure, ...).

You will use a small toolbox called `eg` (the file `eg.py` next to this notebook). The table below lists every command;
each step then shows the one you need, **starting with a cell that prints all the available values** so you always know what you can type.

*You need the address and password of the deployed Neo4j (ask whoever gave you this folder). See `README.md` for setup.*

In [1]:
# pick up edits to eg.py without restarting the kernel
%load_ext autoreload
%autoreload 2
import os
from getpass import getpass
import pandas as pd
import eg      # the toolbox next to this notebook: eg.py

pd.set_option("display.max_rows", 200, "display.max_colwidth", 80)
def table(rows): return pd.DataFrame(rows)   # shows any list of results as a table

## The toolbox: what is in `eg.py`
Every command **takes the words in the brackets** (all optional ones can be left out) and **gives back** either a
result list (wrap it in `table(...)` to see a table) or prints text.

| Command | You give it | You get |
|---|---|---|
| `eg.pilots()` | nothing | each pilot with its number of datasets and variables |
| `eg.datasets(text, pilot)` | optional word, optional pilot | datasets (all, one pilot's, or mentioning a word) |
| `eg.measurement_types(pilot)` | optional pilot | measurement types (Temperature, Pressure, ...) with a short description and how many variables each has |
| `eg.variables(measurement_type, pilot)` | optional measurement type, optional pilot | variables (a sensor or a column) with their unit |
| `eg.find(text, pilot)` | a word, optional pilot | variables whose name or description contains the word |
| `eg.variable(name)` | a variable name | which dataset(s) hold it, its meaning, unit and data lake sensor id |
| `eg.describe(name)` | a variable name | **prints** a card with everything the ontology knows about it |
| `eg.tree(pilot)` | a pilot | **prints** the pilot with its datasets and a few variables each |
| `eg.equipment()` | nothing | each piece of equipment with its pilot and how many variables it hosts |
| `eg.readings_sql(name)` | a variable name | the **SQL text** to fetch its readings from the data lake |
| `eg.connect(uri, user, password)` | address and login | connects to Neo4j (step 0 does this for you) |

One thing to know:
- `pilot` is the short name (`"BER"`, `"REA"`, ...). Step 1 prints all of them.

`table(...)` is not part of the toolbox: it is a one-line helper defined in the cell above.

## 0. Connect
If a `.env` file with the Neo4j settings sits next to this notebook (see `.env.example`), it connects by itself.
Otherwise it asks for the address and password (press Enter to accept the default in brackets).

In [2]:
# Uses the settings in .env (or NEO4J_* environment variables) when present; asks only if it cannot connect.
try:
    eg.connect()
    uri = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
except Exception as e:
    print(f"Could not connect with the saved settings ({type(e).__name__}: {e}); please enter them.")
    default_uri = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
    uri  = input(f"Neo4j address [{default_uri}]: ") or default_uri
    user = os.environ.get("NEO4J_USER") or input("User [neo4j]: ") or "neo4j"
    eg.connect(uri, user, getpass("Password: "))
print("connected to", uri)

connected to bolt://localhost:7687


## 1. Pilots
A **pilot** is a project partner site that provides data. Everything else belongs to one pilot.

**Command:** `eg.pilots()` — takes nothing; returns every pilot with its number of datasets and variables.

In [3]:
# ALL pilots: the names you can pass as `pilot` in the other commands
table(eg.pilots())

,pilot,datasets,variables
0,REA,25,536
1,BER,2,185
2,CARTIF,1,90
3,ENGREEN,6,53
4,CEA,1,45
5,CIEMAT,1,37
6,RDN,5,0


## 2. The whole picture: one pilot as a tree
This is the pilot → dataset → variable structure, for real.

**Command:** `eg.tree(pilot)` — give a pilot; **prints** it with its datasets and the first few variables of each.

In [4]:
pilot = "BER"          # change to any pilot from the table above
eg.tree(pilot)

BER
├── PEM Electrolyzer 14kW  (185 variables)
│   • AS-B103
│   • AS-Z102
│   • AS-Z103
│   • AS-Z104
│   • ... and 181 more
└── PEM Electrolyzer 7kW  (185 variables)
    • AS-B103
    • AS-Z102
    • AS-Z103
    • AS-Z104
    • ... and 181 more


## 3. Datasets
A **dataset** is one collection of data from a pilot, e.g. *PEM Electrolyzer 14kW*. Its title tells you what it is about.

**Command:** `eg.datasets(text=None, pilot=None)` — optionally give a word and/or a pilot; returns the matching datasets
(with the database in the data lake that holds each). No arguments means all of them.

In [5]:
# ALL datasets, grouped by pilot (database = where it sits in the data lake)
table(eg.datasets())

,pilot,dataset,database
0,BER,PEM Electrolyzer 7kW,TEF3_BER
1,BER,PEM Electrolyzer 14kW,TEF3_BER
2,CARTIF,CARTIF hydrogen test bench operational data (15 s),TEF5_CARTIF
3,CEA,CEA MURPHY test bed dataset,TEF4_CEA
4,CIEMAT,CEDER-CIEMAT Microgrid – Daily Operational Data (1 s),TEF2_CEDER
5,ENGREEN,GME CSUD Hourly Electricity Price Dataset,not in the lake
6,ENGREEN,The GSE Historical Metering Dataset,not in the lake
7,ENGREEN,The Antrodoco REC Mapping Dataset,not in the lake
8,ENGREEN,Operational monitoring dataset,TEF7_ENGREEN
9,ENGREEN,The Weather Dataset,TEF7_ENGREEN


In [6]:
# one pilot's datasets, then a word search across all pilots (change the words)
pilot = "BER"
display(table(eg.datasets(pilot=pilot)))
table(eg.datasets("hydrogen"))

,pilot,dataset,database
0,BER,PEM Electrolyzer 7kW,TEF3_BER
1,BER,PEM Electrolyzer 14kW,TEF3_BER


,pilot,dataset,database
0,CARTIF,CARTIF hydrogen test bench operational data (15 s),TEF5_CARTIF


## 4. Measurement types
Every variable has a **measurement type**: Temperature, Pressure, Power, ... The same word means the same thing in every pilot.

**Command:** `eg.measurement_types(pilot=None)` — optionally give a pilot; returns the measurement types with a short description (taken from the ontology) and how many variables each has.

In [7]:
# ALL measurement types (the words you can pass as `measurement_type`), with how many variables each has
table(eg.measurement_types())

,measurement_type,description,variables
0,State,"On/off or discrete operating state, status and alarm signals.",116
1,Power,"Electrical or thermal power (W, kW, MW).",50
2,Control Signal,Control outputs and commands sent to equipment.,38
3,Pressure,"Pressure readings (bar, mbar, Pa).",32
4,Temperature,Temperature readings (°C).,29
5,Timer,"Elapsed or counted time (s, h), such as hour counters.",25
6,Current,Electric current (A).,18
7,Flow,"Flow rates of gases or liquids (L/min, m³/h, kg/h).",18
8,Hydrogen,"Hydrogen-related data: flow, pressure or storage.",10
9,Voltage,Electric voltage (V).,9


In [8]:
# only the measurement types this pilot has
table(eg.measurement_types(pilot))

,measurement_type,description,variables
0,State,"On/off or discrete operating state, status and alarm signals.",64
1,Control Signal,Control outputs and commands sent to equipment.,33
2,Timer,"Elapsed or counted time (s, h), such as hour counters.",25
3,Temperature,Temperature readings (°C).,19
4,Current,Electric current (A).,10
5,Pressure,"Pressure readings (bar, mbar, Pa).",8
6,Flow,"Flow rates of gases or liquids (L/min, m³/h, kg/h).",6
7,Voltage,Electric voltage (V).,4
8,Efficiency,Efficiency indicators.,4
9,Power,"Electrical or thermal power (W, kW, MW).",3


## 5. Variables
A **variable** is a sensor or a column inside a dataset, e.g. `TT-D101`, a storage tank temperature.

**Command:** `eg.variables(measurement_type=None, pilot=None)` — optionally give a measurement type and/or a pilot; returns the variables with their unit.
A variable that appears in two datasets (e.g. BER 14kW and 7kW) gets one row per dataset.

In [9]:
# ALL variables of one pilot (leave out `pilot` for every pilot: over 1,000 rows)
table(eg.variables(pilot=pilot))

,pilot,dataset,variable,measurement_type,unit
0,BER,PEM Electrolyzer 14kW,AS-B103,State,NaN
1,BER,PEM Electrolyzer 14kW,AS-Z102,State,NaN
2,BER,PEM Electrolyzer 14kW,AS-Z103,State,NaN
3,BER,PEM Electrolyzer 14kW,AS-Z104,State,NaN
4,BER,PEM Electrolyzer 14kW,AS-Z105,State,NaN
...,...,...,...,...,...
365,BER,PEM Electrolyzer 7kW,XT-5008,State,word
366,BER,PEM Electrolyzer 7kW,XT-5009,State,word
367,BER,PEM Electrolyzer 7kW,XT-6007,State,word
368,BER,PEM Electrolyzer 7kW,XT-6008,State,word


In [10]:
# variables of one measurement type for that pilot (pilot=None means all pilots)
measurement_type = "Temperature"
table(eg.variables(measurement_type, pilot)).head(15)

,pilot,dataset,variable,measurement_type,unit
0,BER,PEM Electrolyzer 14kW,DT-X201,Temperature,°C
1,BER,PEM Electrolyzer 14kW,TT-4014,Temperature,°C
2,BER,PEM Electrolyzer 14kW,TT-5014,Temperature,°C
3,BER,PEM Electrolyzer 14kW,TT-6014,Temperature,°C
4,BER,PEM Electrolyzer 14kW,TT-B103,Temperature,°C
5,BER,PEM Electrolyzer 14kW,TT-C401,Temperature,°C
6,BER,PEM Electrolyzer 14kW,TT-C901,Temperature,°C
7,BER,PEM Electrolyzer 14kW,TT-D101,Temperature,°C
8,BER,PEM Electrolyzer 14kW,TT-H201,Temperature,°C
9,BER,PEM Electrolyzer 14kW,TT-H405,Temperature,°C


In [11]:
# how many variables of this measurement type does each pilot have?
table(eg.variables(measurement_type)).groupby("pilot").size()

pilot
BER        38
CARTIF      9
ENGREEN     1
dtype: int64

## 6. Find a variable by keyword
Don't know a variable's name? Search the names **and** descriptions of every variable, across all pilots.

**Command:** `eg.find(text, pilot=None)` — give a word (and optionally a pilot); returns the variables whose name, description or
data lake sensor id contains it. Try `pressure`, `storage tank`, `battery`, `wind`, `hydrogen`.

In [12]:
# search by a word (try: pressure, storage tank, battery, wind, H2)
table(eg.find("storage tank"))

,pilot,dataset,variable,meaning,unit,measurement_type
0,BER,PEM Electrolyzer 14kW,LT-D101,Storage tank level,L,Volume
1,BER,PEM Electrolyzer 14kW,PT-D101,Storage tank bottom press,mbarg,Pressure
2,BER,PEM Electrolyzer 14kW,TT-D101,Storage tank temperature,°C,Temperature
3,BER,PEM Electrolyzer 7kW,LT-D101,Storage tank level,L,Volume
4,BER,PEM Electrolyzer 7kW,PT-D101,Storage tank bottom press,mbarg,Pressure
5,BER,PEM Electrolyzer 7kW,TT-D101,Storage tank temperature,°C,Temperature


## 7. One variable in detail
What is it, and what is its id in the data lake? The same name can appear in several datasets, one row each.

**Commands:** `eg.variable(name)` — give a variable name; returns where it lives (dataset, meaning, unit, data lake sensor id).
`eg.describe(name)` — give a variable name; **prints** a card with everything the ontology knows about it.

In [13]:
name = "TT-D101"      # pick any name from step 5 or 6
table(eg.variable(name))

,pilot,dataset,meaning,unit,data_lake_sensor_id
0,BER,PEM Electrolyzer 7kW,Storage tank temperature,°C,TT_D101
1,BER,PEM Electrolyzer 14kW,Storage tank temperature,°C,TT_D101


In [14]:
eg.describe(name)

== TT-D101  (BER, dataset: PEM Electrolyzer 7kW)
   Location Soria, Spain | period - | updated DAILY
   Meaning         Storage tank temperature
   Measurement type Temperature
   Unit            °C
   Valid range     0.0 to 100.0
   Kind            measured
   Data type       float
   Sampling        Over threshold (0.1s) or 30s
   Aggregation     instantaneous
   Equipment       feedwater
   Data lake sensor id TT_D101
   Data lake match normalised
   Database        TEF3_BER
   Table           public.f_tsdata
== TT-D101  (BER, dataset: PEM Electrolyzer 14kW)
   Location Genova, Italy | period 2026-06-18 - Ongoing | updated DAILY
   Meaning         Storage tank temperature
   Measurement type Temperature
   Unit            °C
   Valid range     0.0 to 100.0
   Kind            measured
   Data type       float
   Sampling        Over threshold (0.1s) or 30s
   Aggregation     instantaneous
   Equipment       feedwater
   Data lake sensor id TT_D101
   Data lake match normalised
   Dat

## 8. The SQL for its readings
**Command:** `eg.readings_sql(name, n=10)` — give a variable name (and optionally how many rows); returns the **SQL text** that fetches
its latest readings. Run it in the pilot's database of the data lake (this notebook doesn't connect to the lake).

In [15]:
print(eg.readings_sql(name))

SELECT c.datetime, f.f_value
FROM public.f_tsdata f JOIN public.d_calendar c USING (calendar_id)
WHERE f.sensor_id = 'TT_D101'
ORDER BY f.calendar_id DESC LIMIT 10;


## 9. Recipes
Three everyday questions, answered by combining the commands above. Copy one and change the words.

**Which pilots have pressure variables on the hydrogen (H2) line?**  
`find` searches names and descriptions; the `measurement_type` column lets us keep only pressure.

In [16]:
h2 = table(eg.find("H2", limit=300))
h2[h2["measurement_type"] == "Pressure"][["pilot", "variable", "meaning", "unit"]].drop_duplicates()

,pilot,variable,meaning,unit
10,BER,PT-H101,H2 line press before dryer,bar
11,BER,PT-H301,H2 line press after dryer,bar
36,CARTIF,ELE_Inner_H2_Pressure,Pressure measurement at the electrolyzer inlet,bar
37,CARTIF,ELE_Outer_H2_Pressure,Pressure measurement at the electrolyzer outlet,bar


**List every temperature variable of one pilot with its unit.**  
Each variable appears once per dataset, so we drop the duplicates.

In [17]:
table(eg.variables("Temperature", "BER"))[["variable", "unit"]].drop_duplicates()

,variable,unit
0,DT-X201,°C
1,TT-4014,°C
2,TT-5014,°C
3,TT-6014,°C
4,TT-B103,°C
5,TT-C401,°C
6,TT-C901,°C
7,TT-D101,°C
8,TT-H201,°C
9,TT-H405,°C


**Where does `PT-D101` live in the data lake, and how do I fetch its readings?**  
The data lake sensor id and database come from `variable`; `readings_sql` turns them into SQL to run in the lake.

In [18]:
display(table(eg.variable("PT-D101")))
print(eg.readings_sql("PT-D101"))

,pilot,dataset,meaning,unit,data_lake_sensor_id
0,BER,PEM Electrolyzer 7kW,Storage tank bottom press,mbarg,PT_D101
1,BER,PEM Electrolyzer 14kW,Storage tank bottom press,mbarg,PT_D101


SELECT c.datetime, f.f_value
FROM public.f_tsdata f JOIN public.d_calendar c USING (calendar_id)
WHERE f.sensor_id = 'PT_D101'
ORDER BY f.calendar_id DESC LIMIT 10;


## Extras: equipment (optional)
Variables of BER, CEA, CARTIF, CIEMAT and ENGREEN are hosted by a piece of **equipment** (a tank, a dryer, a battery) (as they wrote it, so spellings vary).

**Command:** `eg.equipment()` — takes nothing; returns every piece of equipment with how many variables it hosts.

In [19]:
# ALL equipment, with how many variables each
table(eg.equipment())

,pilot,equipment,variables
0,BER,dryer,26
1,BER,psu,25
2,BER,feedwater,25
3,BER,HourCounter,25
4,BER,system,24
5,BER,h2,18
6,BER,safety,12
7,BER,power,11
8,BER,cooling,7
9,BER,control,6
